In [1]:
from pathlib import Path

import nibabel as nib
import numpy as np

t1w_image_path = Path(
    "/scratch/imaging/ds005237-fmriprep/sub-NDARINVXP963GZ5/anat/sub-NDARINVXP963GZ5_run-01_desc-preproc_T1w.nii.gz"
)
bold_image_path = Path(
    "/scratch/imaging/ds005237/sub-NDARINVXP963GZ5/func/sub-NDARINVXP963GZ5_task-restPA_run-01_bold.nii.gz"
)

t1w_image = nib.nifti1.load(t1w_image_path)  # pyright: ignore[reportAttributeAccessIssue]
bold_image = nib.nifti1.load(bold_image_path)  # pyright: ignore[reportAttributeAccessIssue]

In [2]:
from nilearn import image

# Use the first volume of the raw BOLD time series
bold_volume = image.index_img(bold_image, 0)

# Zoom in on the brain using the bounding box of the brain mask in world coordinates
brain_mask_image = nib.nifti1.load(  # pyright: ignore[reportAttributeAccessIssue]
    t1w_image_path.parent / "sub-NDARINVXP963GZ5_run-01_desc-brain_mask.nii.gz"
)
brain_coords = nib.affines.apply_affine(
    brain_mask_image.affine, np.argwhere(brain_mask_image.get_fdata() > 0)
)
# Add 10% padding on each side
minimum, maximum = brain_coords.min(axis=0), brain_coords.max(axis=0)
padding = 0.1 * (maximum - minimum)
x_limits = (minimum[0] - padding[0], maximum[0] + padding[0])
z_limits = (minimum[2] - padding[2], maximum[2] + padding[2])
x_limits, z_limits

((np.float64(-79.28805932067334), np.float64(82.4822308499366)),
 (np.float64(-64.5641587043181), np.float64(84.49809160660952)))

In [3]:
import scipy.ndimage
from PIL import Image, ImageChops
from nilearn import plotting

# Compute a head mask from the T1w image, so that only the air around the head is hidden
t1w_data = t1w_image.get_fdata()
head_mask = t1w_data > 0.1 * np.percentile(t1w_data[t1w_data > 0], 99)
head_mask = scipy.ndimage.binary_closing(head_mask, iterations=3)
head_mask = scipy.ndimage.binary_fill_holes(head_mask)
head_mask = scipy.ndimage.binary_opening(head_mask, iterations=2)
labels, _ = scipy.ndimage.label(head_mask)
head_mask = labels == np.argmax(np.bincount(labels[head_mask]))
head_mask_image = image.new_img_like(t1w_image, head_mask.astype(np.uint8))


def mask_head(plot_image):
    data = plot_image.get_fdata()
    mask = image.resample_to_img(head_mask_image, plot_image, interpolation="nearest").get_fdata() > 0
    # Erode to remove the ragged edge of the head
    mask = scipy.ndimage.binary_erosion(mask, iterations=2)
    # Keep dark voxels inside the head, such as the ventricles, above the threshold
    scale = np.percentile(data[mask], 99)
    data = np.where(mask, np.maximum(data, 0.01 * scale), 0.0)
    return image.new_img_like(plot_image, data), 0.005 * scale


for name, plot_image in [("t1w-slice.png", t1w_image), ("bold-slice.png", bold_volume)]:
    masked_image, threshold = mask_head(plot_image)
    display = plotting.plot_img(
        masked_image,
        display_mode="y",
        cut_coords=[0],
        cmap="gray",
        threshold=threshold,
        # Nearest-neighbor interpolation avoids ringing at the edge of the mask
        resampling_interpolation="nearest",
        vmin=0,
        annotate=False,
        draw_cross=False,
        colorbar=False,
        black_bg=False,
    )
    (axes,) = display.axes.values()
    axes.ax.set_xlim(*x_limits)
    axes.ax.set_ylim(*z_limits)
    display.frame_axes.figure.savefig(name, dpi=300, bbox_inches="tight", facecolor="white")
    display.close()

    # Crop the white margin around the head
    with Image.open(name) as png:
        png = png.convert("RGB")
        bbox = ImageChops.invert(png).getbbox()
        png.crop(bbox).save(name)